# System 2 Failure-Mode Triage (Phase 1 of Capability-Triage Plan)

**Zweck:** Auf der bestehenden S2-Baseline (`data/results/sys2_baseline_raw_20260419_120053.json`, 77 Queries auf Gold-Standard v2) eine klassifizierte Verteilung der Fehlerursachen erzeugen. Damit wird das Decision-Gate des Capability-Triage-Plans befuellt: Dominiert Math (`math_error` + `wrong_number_right_chunk`) den FA-3-Fehleranteil mit ≥ 30 % der FA-3-Failures, so geht Phase 2 in Variante 2A (`compute_growth`-Tool einfuehren); andernfalls 2B (kein neues Tool, Verweis auf Retrieval-Bug-Backlog).

**Methodisch wichtig**: Kein neuer Eval-Lauf, kein RAGAS, keine Kosten. Wir nutzen die existierenden Tool-Trails und Antworten und klassifizieren rein post-hoc.

## Failure-Kategorien

| Kategorie | Definition |
|---|---|
| `correct` | Antwort enthaelt den Ground-Truth-Wert mit Toleranz (relative 5 % bei Zahlen, exakter Match bei Strings) und keine Loop-Erschoepfung |
| `recursion_exhausted` | Antwort ist der LangGraph-Marker `"Sorry, need more steps to process this request."` |
| `retrieval_miss` | Antwort enthaelt typische Refusal-Phrasen (`"konnte ... nicht finden"`, `"nicht verfuegbar"`, `"nicht aufgefuehrt"`, `"keine relevanten"`, `"nicht enthalten"`) und kein numerischer Match |
| `wrong_number_right_chunk` | Antwort numerisch falsch, aber der GT-Wert ist in den `contexts` (Tool-Outputs) enthalten — d. h. der Agent hatte die richtigen Zahlen gesehen, aber falsch verwendet |
| `wrong_concept` | Antwort numerisch falsch und GT-Wert NICHT in den Contexts — Agent hat die falsche Stelle gewaehlt oder das falsche Konzept extrahiert |
| `math_error` | Antwort enthaelt eine sichtbare Berechnung mit falschem Ergebnis trotz korrekter Inputs (manuelle Inspektion noetig — wird in `wrong_number_right_chunk` als Obermenge mitgezaehlt fuer das Gate, weil `compute_growth` beide adressieren wuerde) |
| `other` | Restkategorie |

## Decision-Gate

`math_share_in_fa3 = (math_error + wrong_number_right_chunk) / total_fa3_failures` — Schwelle 30 %.

In [ ]:
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation.gold_standard_loader import load_gold_standard

S2_RAW = project_root / "data" / "results" / "sys2_baseline_raw_20260419_120053.json"
GOLD_V2 = project_root / "notebooks" / "experiments" / "sys1_rag_monolith" / "ablation_test_data_v2.csv"
OUT_CSV = project_root / "data" / "results" / "sys2_failure_triage.csv"

print("Project root:", project_root)
print("S2 raw exists:", S2_RAW.exists())
print("Gold v2 exists:", GOLD_V2.exists())

In [ ]:
with S2_RAW.open() as f:
    s2_runs = json.load(f)

gold_items = {it.id: it for it in load_gold_standard(GOLD_V2)}

print(f"S2 runs: {len(s2_runs)}")
print(f"Gold items (v2): {len(gold_items)}")

missing_in_gold = [r['id'] for r in s2_runs if r['id'] not in gold_items]
missing_in_s2 = [i for i in gold_items if i not in {r['id'] for r in s2_runs}]
print(f"In S2 but not Gold: {missing_in_gold}")
print(f"In Gold but not S2: {missing_in_s2}")

In [ ]:
NUMBER_RE = re.compile(r"-?\d+(?:[.,]\d+)*")

REFUSAL_PHRASES = (
    "konnte nicht finden",
    "konnte ... nicht finden",
    "nicht verfuegbar",
    "nicht verf\u00fcgbar",
    "nicht aufgef\u00fchrt",
    "nicht aufgefuehrt",
    "keine relevanten",
    "nicht enthalten",
    "nicht explizit",
    "konnte nicht ermitteln",
    "konnte ich nicht",
    "nicht gefunden",
    "nicht direkt aufgef",
    "nicht ermitteln",
    "liefern keine",
    "lieferten keine",
    "lieferte keine",
    "nicht in den verf",
    "nicht im bereitgestellten kontext",
    "nicht beantwortet werden",
    "nicht berechnet werden",
    "daher kann",
    "nicht direkt ergibt",
    "daten konnten",
    "sorry, need more steps",
)
RECURSION_MARKER = "sorry, need more steps to process this request."


def parse_number(value: str) -> float | None:
    """Extract the first plausible numeric token from a string.

    Heuristic: matches an integer or decimal number with optional thousands
    separators (`.` or `,`). Handles common German formats like `391.035`
    (thousands) and `30,8` (decimal comma) as well as US format `1,697.31`.
    """
    if not value:
        return None
    m = NUMBER_RE.search(value)
    if not m:
        return None
    raw = m.group(0).replace(" ", "")
    if "," in raw and "." in raw:
        if raw.rfind(",") > raw.rfind("."):
            raw = raw.replace(".", "").replace(",", ".")
        else:
            raw = raw.replace(",", "")
    elif "," in raw:
        if len(raw.split(",")[-1]) in (1, 2, 3) and len(raw.split(",")[-1]) != 3:
            raw = raw.replace(",", ".")
        else:
            raw = raw.replace(",", "")
    try:
        return float(raw)
    except ValueError:
        return None


def gt_in_text(gt_value: str, text: str, rel_tol: float = 0.05) -> bool:
    """Return True if any numeric token in the GT appears in text within rel_tol.

    Falls GT keine Zahl enthaelt, faellt die Funktion auf Case-insensitive
    Substring-Match zurueck (z. B. fuer qualitative GTs wie 'Leicht steigend').
    """
    if not gt_value or not text:
        return False
    gt_num = parse_number(gt_value)
    if gt_num is None:
        return gt_value.strip().lower() in text.lower()
    # Magnitude families: scale GT and search for any matching scale in text.
    candidates = [gt_num]
    if "mrd" in gt_value.lower() or "billion" in gt_value.lower():
        candidates += [gt_num * 1000]
    if "mio" in gt_value.lower() or "million" in gt_value.lower():
        candidates += [gt_num / 1000]
    text_nums = [parse_number(tok) for tok in NUMBER_RE.findall(text)]
    text_nums = [n for n in text_nums if n is not None]
    for c in candidates:
        if c == 0:
            continue
        for n in text_nums:
            if abs(n - c) / abs(c) <= rel_tol:
                return True
    return False


def contains_refusal(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in REFUSAL_PHRASES)


def is_recursion_exhausted(answer: str) -> bool:
    return RECURSION_MARKER in answer.strip().lower()

In [ ]:
def classify(run: dict, gt: str) -> str:
    answer = run.get("answer") or ""
    contexts = run.get("contexts") or []
    contexts_blob = "\n".join(c if isinstance(c, str) else str(c) for c in contexts)

    if is_recursion_exhausted(answer):
        return "recursion_exhausted"

    if gt_in_text(gt, answer):
        return "correct"

    if contains_refusal(answer) and not gt_in_text(gt, answer):
        return "retrieval_miss"

    if gt_in_text(gt, contexts_blob):
        return "wrong_number_right_chunk"

    return "wrong_concept"


rows = []
for run in s2_runs:
    rid = run["id"]
    gold = gold_items.get(rid)
    if gold is None:
        continue
    category = classify(run, gold.ground_truth)
    rows.append({
        "id": rid,
        "fa_type": gold.fa_type,
        "query_type": gold.query_type,
        "doc_refs": gold.doc_refs,
        "question": gold.question,
        "ground_truth": gold.ground_truth,
        "answer": (run.get("answer") or "")[:240],
        "num_steps": run.get("num_steps"),
        "tool_calls": ",".join(run.get("tool_calls") or []),
        "reflection_issues": ",".join(run.get("reflection_issues") or []),
        "category": category,
    })

df = pd.DataFrame(rows)
df.head()

In [ ]:
# Manuelle Overrides fuer Faelle, in denen die Heuristik bekanntermassen schief geht.
# Begruendung pro Override im Side-by-Side ([data/results/sys1_vs_sys2_v2_comparison.md])
# verifiziert. Konservativ gehalten — nur eindeutige Faelle.
MANUAL_OVERRIDES: dict[int, str] = {
    # Query 3: Debt-to-Equity AAPL FY24. GT=540.9% (Total Liabilities/Equity).
    # S2 gab 169.7% (Term Debt/Equity). Beide Zahlen aus dem Filing valide,
    # aber konzeptuell andere Definition.
    3: "wrong_concept",
    # Query 18: Debt/Equity GOOGL FY24. GT=44.5%. S2 gab 3.3% (Long-term debt only).
    18: "wrong_concept",
    # Query 21: Operating Margin AAPL FY24. GT=31.5%. S2 gab 41.44%
    # (falsche Operating-Income-Zeile, aber GT-Wert nicht im Chunk-Blob).
    21: "wrong_concept",
    # Query 24: Operating Margin AAPL FY23. GT=30.8%. S2 gab 29.82% — Berechnung
    # mit falscher Operating-Income-Zahl, GT nicht in Chunks.
    24: "wrong_concept",
    # Query 32: AWS Operating Margin AMZN FY24. GT=29.6%. S2 gab 17.47%.
    # Falsche Segment-Zahl genutzt.
    32: "wrong_concept",
    # Query 41 (FA-3): YoY-aehnlich. S2 gab -0.835 %. GT -0.8 %. Match-Toleranz.
    41: "correct",
    # Query 43 (FA-3): GT '-3.4 %'. S2 gab -$3.259 Mio absolute Diff statt %.
    # Konzeptuell: hat absolute statt prozentual berechnet -> wrong_concept.
    43: "wrong_concept",
    # Query 45 (FA-3): GT '+$3.8 Mrd'. S2 gab +$5.119 Mio.
    # Hat falsche FY-22-Zahl gewaehlt (26.251 statt 27.572). Chunks nicht eindeutig.
    45: "wrong_concept",
    # Query 50: GT '+7,000'. S2 gab '7.000' (richtig). Match-Toleranz Zahl-Format.
    50: "correct",
    # Query 53: GT '+$28.9 Mrd'. S2 gab 28.823 Mio (richtig im Mio-Massstab).
    53: "correct",
    # Query 55: GT '+$12 Mrd'. S2 gab '228 Mio' (Fulfillment != Operating Expenses).
    # Zahlen aus Filing, aber falsche Zeile.
    55: "wrong_concept",
    # Query 57: GT 14.7 %. S2 gab 13.87 % — abweichend, weil GT auf USD-Basis statt
    # auf disaggregated revenue; numerisch nahe aber > 5 % rel — Toleranz haelt es
    # auf wrong_concept. Bleibt automatisch.
    # Query 70: FY24 GROSS MARGIN. GT '~56% > 44%'. S2 traf grob richtig.
    70: "correct",
    # Query 71: 'Sorry, need more steps' -> bleibt automatisch recursion_exhausted.
    # Query 75: GT 'AMZN > AAPL'. S2 sagte Apple>Amazon faelschlich -> wrong_concept.
    75: "wrong_concept",
}

df["category_auto"] = df["category"]
df["category"] = df.apply(
    lambda r: MANUAL_OVERRIDES.get(r["id"], r["category"]), axis=1,
)
df["overridden"] = df["category"] != df["category_auto"]
print(f"Manual overrides applied: {df['overridden'].sum()}")
df[df["overridden"]][["id", "fa_type", "category_auto", "category", "ground_truth"]]

In [ ]:
overall = df["category"].value_counts().to_frame("count")
overall["share"] = (overall["count"] / overall["count"].sum()).round(3)
overall

In [ ]:
by_fa = (
    df.groupby(["fa_type", "category"]).size().unstack(fill_value=0)
)
by_fa["total"] = by_fa.sum(axis=1)
by_fa

In [ ]:
fa3 = df[df["fa_type"] == "FA-3"]
fa3_total = len(fa3)
fa3_failures = fa3[fa3["category"] != "correct"]
fa3_failure_total = len(fa3_failures)
fa3_math = fa3_failures[fa3_failures["category"].isin(["wrong_number_right_chunk", "math_error"])]
fa3_math_count = len(fa3_math)

math_share = fa3_math_count / fa3_failure_total if fa3_failure_total else 0.0

print("=" * 64)
print("DECISION GATE for Phase 2")
print("=" * 64)
print(f"FA-3 total queries: {fa3_total}")
print(f"FA-3 failures (category != correct): {fa3_failure_total}")
print(f"FA-3 math-adjacent failures (wrong_number_right_chunk + math_error): {fa3_math_count}")
print(f"Math share in FA-3 failures: {math_share:.1%}")
print()
threshold = 0.30
verdict = "Phase 2A (compute_growth)" if math_share >= threshold else "Phase 2B (no new tool)"
print(f"Threshold: {threshold:.0%}")
print(f"VERDICT: {verdict}")
print()
print("Auch zusaetzliche Sicht: FA-3 Hauptfailure-Kategorie")
print(fa3_failures["category"].value_counts())

In [ ]:
df_out = df[[
    "id", "fa_type", "query_type", "doc_refs",
    "question", "ground_truth", "answer",
    "tool_calls", "num_steps", "reflection_issues",
    "category_auto", "category", "overridden",
]]
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(OUT_CSV, sep=";", index=False)
print(f"Wrote {len(df_out)} rows to {OUT_CSV}")

In [ ]:
lines = [
    "Failure-Mode Triage Summary",
    "=" * 64,
    f"Source: {S2_RAW.name}",
    f"Gold: {GOLD_V2.name} (v2, 77 entries)",
    "",
    "Overall distribution:",
]
for cat, count in df["category"].value_counts().items():
    share = count / len(df)
    lines.append(f"  {cat:<28s} {count:>3d}  ({share:.1%})")
lines += ["", "By FA type:"]
for fa in sorted(df["fa_type"].unique()):
    sub = df[df["fa_type"] == fa]
    lines.append(f"  {fa} (n={len(sub)}):")
    for cat, count in sub["category"].value_counts().items():
        share = count / len(sub) if len(sub) else 0
        lines.append(f"    {cat:<28s} {count:>3d}  ({share:.1%})")
lines += [
    "",
    f"Decision gate: math share in FA-3 failures = {math_share:.1%} "
    f"(threshold {threshold:.0%}) -> {verdict}",
]
print("\n".join(lines))

## Interpretation und Konsequenz

Die ausgegebene `VERDICT`-Zeile entscheidet, ob im naechsten Schritt das
`compute_growth`-Tool eingefuehrt wird (`Phase 2A`) oder ob ein dokumentierter
Verzicht in Form eines Decision-Log-Eintrags erfolgt (`Phase 2B`).

**Caveats**:
- Die Klassifikation ist heuristisch. `MANUAL_OVERRIDES` deckt die im Side-by-Side eindeutig identifizierten Faelle ab; ein restlicher Klassifikationsfehler von ~5–10 % auf den 77 Items ist nicht ausgeschlossen.
- Die `tool_calls`-Liste im S2-Raw enthaelt nur die Tool-Namen, nicht Argumente oder Outputs einzelner `calculate`-Aufrufe. Echte `math_error`-Vorfaelle (`calculate` mit korrekten Inputs aber falschem Output) sind daher unterspezifiziert; wir zaehlen sie konservativ zu `wrong_number_right_chunk`, weil `compute_growth` beide adressieren wuerde.
- MSFT-Item-8-Retrieval-Misses sind durch den bekannten Ingestion-Bug ([BACKLOG.md 2026-05-10](../../docs/decisions/BACKLOG.md)) systematisch erzeugt — sie schlagen in `retrieval_miss` zu Buche und sind nicht durch ein neues Math-Tool behebbar.